In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import paths   # input/output locations: set SN2024VJM_DATA / SN2024VJM_EXTERNAL (see README)
from   astropy import stats
from   astropy import units as u
# from   astropy.cosmology import Planck18 as cosmo
# from   astropy.cosmology import Planck15 as cosmo
from   astropy import units as u
import extinction
import glob
from   mpdaf.obj import Cube
# add the folder to path
import numpy as np
import matplotlib.pyplot as plt
from   astropy import table
# eml_tools (balmer_decrement, resample, ebvCor) must be importable, e.g. on PYTHONPATH
import eml_tools
from   scipy.optimize import fsolve

%matplotlib widget

In [ ]:
lines = {
    'H-a':          6562.80,
    'H-b':          4862.69,
    'H-g':          4341.69,
    # 'H-d':          4102.92,
    # '[OII]-3726':   3726.09,
    # '[OII]-3729':   3729.88,
    '[OIII]-5008':  5008.24,
    '[OIII]-4960':  4960.30,
    '[NII]-6547':   6547.96,
    '[NII]-6583':   6583.34,
    '[SII]-6716':   6716.31,
    '[SII]-6732':   6730.68,
}

def gauss_model(x, A, xc, w):
    return A / (w * np.sqrt(np.pi/2.)) * np.exp( -2 * (x-xc)**2/w**2)

def measure_eml(DATA, REDSHIFT):

    result = table.Table(dtype=('S100', 'f', 'f', 'f', 'f', 'f', 'f', 'f', 'f'),
                     names=('LINE', 'FLUX', 'REDSHIFT', 'SIGMA', 'FLUX_ERR', 'REDSHIFT_ERR', 'SIGMA_ERR', 'MU', 'MU_ERR')
                     )

    plt.figure(figsize=(9*2, 9))
    ax=plt.subplot(111)

    ax.plot(DATA['WAVE_REST'], DATA['FLAMBDA_EXTCORR'] , color='black',   drawstyle='steps')

    white_list = ['H-a','H-b', '[OIII]-5008', '[OIII]-4960']

    for key in lines.keys():
        alpha  = 1 if key in white_list else 0.5
        zorder = 10  if key in white_list else 0
        ax.plot(lines[key], 0.9*max(DATA['FLAMBDA_EXTCORR']), marker='v', color='tab:blue', ms=9, zorder=zorder, alpha=alpha)

    #######

    labels = ['[OIII]-4960', '[OIII]-5008']
    line = [lines[x] for x in labels]

    line_1 = min(line)
    line_2 = max(line)
    buffer_cont = 20
    buffer_lines = 5

    mask_region = (DATA['WAVE_REST'] > line_1 - buffer_cont) & (DATA['WAVE_REST'] < line_2 + buffer_cont)

    mask_line_1 = np.abs(DATA['WAVE_REST'] - line_1 ) < buffer_lines
    mask_line_2 = np.abs(DATA['WAVE_REST'] - line_2 ) < buffer_lines
    mask_lines  = np.logical_or(mask_line_1, mask_line_2)

    mask_cont   = np.logical_xor(mask_region, mask_lines)

    try:
        def fitfunc (x, A, B, C, D, A0, REDSHIFT, W0):

            return A * x**3 + B * x**2 + C * x + D + gauss_model(x, A0, (1 + REDSHIFT) * line[0], W0) +\
                                                     gauss_model(x, 3*A0, (1 + REDSHIFT) * line[1], W0)

        pinit        = [0, 0, 0, 0,
                        np.nanmax(DATA['FLAMBDA_EXTCORR'][mask_region]),
                        0,
                        3]

        from scipy.optimize import curve_fit
        popt, pcov     = curve_fit(fitfunc,
                                DATA['WAVE_REST'][mask_region],
                                DATA['FLAMBDA_EXTCORR'][mask_region],
                                sigma  = DATA['FLAMBDA_EXTCORR'][mask_region]*0.1,
                                p0     = pinit,
                                bounds = ([-np.inf, -np.inf, -np.inf, -np.inf, 0, -0.1, 2.9],
                                          [ np.inf,  np.inf,  np.inf,  np.inf, np.inf, 0.1, 4.9]))

        perr = np.sqrt(np.diag(pcov))

        result.add_row([labels[0], popt[4],   popt[5], popt[6],   perr[4], perr[5], perr[6], (1 + popt[5]) * line[0], perr[5] * line[0]])
        result.add_row([labels[1], 3*popt[4], popt[5], popt[6], 3*perr[4], perr[5], perr[6], (1 + popt[5]) * line[0], perr[5] * line[1]])

        width = popt[6]

    except:
        result.add_row([labels[0], np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan])
        result.add_row([labels[1], np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan])


    plt.figure(figsize=(5*np.sqrt(2), 5))
    ax = plt.subplot(111)

    ax.plot(DATA['WAVE_REST'][mask_region], DATA['FLAMBDA_EXTCORR'][mask_region])
    ax.plot(DATA['WAVE_REST'][mask_region], DATA['FLAMBDA_EXTCORR'][mask_region]*0.1, color='tab:green')

    try:
        ax.plot(DATA['WAVE_REST'][mask_region], fitfunc(DATA['WAVE_REST'][mask_region], *popt*1e-20))
    except:
        pass

    labels = ['H-b']
    line = [lines[x] for x in labels]

    line_1 = min(line)
    line_2 = max(line)
    buffer_cont = 20
    buffer_lines = 5

    mask_region = (DATA['WAVE_REST'] > line_1 - buffer_cont) & (DATA['WAVE_REST'] < line_2  + buffer_cont)

    mask_line_1 = np.abs(DATA['WAVE_REST'] - line_1 ) < buffer_lines
    #mask_line_2 = np.abs(data_xsh_wo_extcorr['vac_helio_wave'] - line_2 * (1+sn2021yfj_dict['redshift']) ) < buffer_lines
    mask_lines  = np.logical_or(mask_line_1, mask_line_2)

    mask_cont   = np.logical_xor(mask_region, mask_lines)

    try:
        pinit        = [0, 0, 0, 0,
                        np.nanmax(DATA['FLAMBDA_EXTCORR'][mask_region]),
                        0]

        def fitfunc (x, A, B, C, D, A0, REDSHIFT):
            W0=width
            return A * x**3 + B * x**2 + C * x + D + gauss_model(x, A0, (1 + REDSHIFT) * line[0], W0)

        from scipy.optimize import curve_fit
        popt, pcov     = curve_fit(fitfunc,
                                DATA['WAVE_REST'][mask_region],
                                DATA['FLAMBDA_EXTCORR'][mask_region],
                                sigma=DATA['FLAMBDA_EXTCORR'][mask_region]*0.1,
                                p0=pinit,
                                bounds = ([-np.inf, -np.inf, -np.inf, -np.inf, 0, -0.1],
                                          [ np.inf,  np.inf,  np.inf,  np.inf, np.inf, 0.1])
                                )


        perr = np.sqrt(np.diag(pcov))

        result.add_row([labels[0], popt[4], popt[5], width, perr[4], perr[5], 0, (1 + popt[5]) * line[0], perr[5] * line[0]])

    except:
        result.add_row([labels[0], np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan])

    plt.figure(figsize=(5*np.sqrt(2), 5))
    ax = plt.subplot(111)

    ax.plot(DATA['WAVE_REST'][mask_region], DATA['FLAMBDA_EXTCORR'][mask_region])
    ax.plot(DATA['WAVE_REST'][mask_region], DATA['FLAMBDA_EXTCORR'][mask_region]*0.1, color='tab:green')

    try:
        ax.plot(DATA['WAVE_REST'][mask_region], fitfunc(DATA['WAVE_REST'][mask_region], *popt))
    except:
        pass

    labels = ['H-g']
    line = [lines[x] for x in labels]

    line_1 = min(line)
    line_2 = max(line)
    buffer_cont = 20
    buffer_lines = 5

    mask_region = (DATA['WAVE_REST'] > line_1 - buffer_cont) & (DATA['WAVE_REST'] < line_2  + buffer_cont)

    mask_line_1 = np.abs(DATA['WAVE_REST'] - line_1 ) < buffer_lines
    #mask_line_2 = np.abs(data_xsh_wo_extcorr['vac_helio_wave'] - line_2 * (1+sn2021yfj_dict['redshift']) ) < buffer_lines
    mask_lines  = np.logical_or(mask_line_1, mask_line_2)

    mask_cont   = np.logical_xor(mask_region, mask_lines)

    try:
        pinit        = [0, 0, 0, 0,
                        np.nanmax(DATA['FLAMBDA_EXTCORR'][mask_region]),
                        0]

        def fitfunc (x, A, B, C, D, A0, REDSHIFT):
            W0 = width
            return A * x**3 + B * x**2 + C * x + D + gauss_model(x, A0, (1 + REDSHIFT) * line[0], W0)

        from scipy.optimize import curve_fit
        popt, pcov     = curve_fit(fitfunc,
                                DATA['WAVE_REST'][mask_region],
                                DATA['FLAMBDA_EXTCORR'][mask_region],
                                sigma=DATA['FLAMBDA_EXTCORR'][mask_region]*0.1,
                                p0=pinit,
                                bounds = ([-np.inf, -np.inf, -np.inf, -np.inf, 0, -0.1],
                                          [ np.inf,  np.inf,  np.inf,  np.inf, np.inf, 0.1])
                                )

        perr = np.sqrt(np.diag(pcov))
        result.add_row([labels[0], popt[4], popt[5], width, perr[4], perr[5], 0, (1 + popt[5]) * line[0], perr[5] * line[0]])
    
    except:
        result.add_row([labels[0], np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan])

    plt.figure(figsize=(5*np.sqrt(2), 5))
    ax = plt.subplot(111)

    ax.plot(DATA['WAVE_REST'][mask_region], DATA['FLAMBDA_EXTCORR'][mask_region])
    ax.plot(DATA['WAVE_REST'][mask_region], DATA['FLAMBDA_EXTCORR'][mask_region]*0.1, color='tab:green')

    try:
        ax.plot(DATA['WAVE_REST'][mask_region], fitfunc(DATA['WAVE_REST'][mask_region], *popt))
    except:
        pass

    labels = ['H-a', '[NII]-6547', '[NII]-6583']
    line = [lines[x] for x in labels]

    line_1 = min(line)
    line_2 = max(line)
    buffer_cont = 25
    buffer_lines = 7

    mask_region = (DATA['WAVE_REST'] > line_1 - buffer_cont) & (DATA['WAVE_REST'] < line_2 + buffer_cont)

    mask_line_1 = np.abs(DATA['WAVE_REST'] - line_1 ) < buffer_lines
    mask_line_2 = np.abs(DATA['WAVE_REST'] - line_2 ) < buffer_lines
    mask_lines  = np.logical_or(mask_line_1, mask_line_2)

    mask_cont   = np.logical_xor(mask_region, mask_lines)

    try:
        def fitfunc (x, A, B, C, D, A0, A1, REDSHIFT, W0):
            #W0 =4.4

            return 0*A * x**3 + B * x**2 + C * x + D + gauss_model(x, A0,   (1 + REDSHIFT) * line[0], W0) +\
                                                     gauss_model(x, A1,   (1 + REDSHIFT) * line[1], W0) +\
                                                     gauss_model(x, 3*A1,   (1 + REDSHIFT) * line[2], W0)

        pinit        = [0, 0, 0, 0,
                        np.nanmax(DATA['FLAMBDA_EXTCORR'][mask_region]),
                        np.nanmax(DATA['FLAMBDA_EXTCORR'][mask_region]),
                        0,
                        3
                        ]

        from scipy.optimize import curve_fit
        popt, pcov     = curve_fit(fitfunc,
                                DATA['WAVE_REST'][mask_region],
                                DATA['FLAMBDA_EXTCORR'][mask_region],
                                sigma  = DATA['FLAMBDA_EXTCORR'][mask_region]*0.1,
                                p0     = pinit,
                                bounds = ([-np.inf, -np.inf, -np.inf, -np.inf, 0, 0, -0.1, 1.9],
                                          [ np.inf,  np.inf,  np.inf,  np.inf, np.inf, np.inf, 0.1, 5.9]))
#                                bounds = ([-np.inf, -np.inf, -np.inf, -np.inf, 0, 0, -0.1],
#                                          [ np.inf,  np.inf,  np.inf,  np.inf, np.inf, np.inf, 0.1]))

        perr = np.sqrt(np.diag(pcov))

        result.add_row([labels[0],   popt[4], popt[6], popt[7], perr[4],   perr[6], perr[7], (1 + popt[6]) * line[0], perr[6] * line[0]])
        result.add_row([labels[1],   popt[5], popt[6], popt[7], perr[5],   perr[6], perr[7], (1 + popt[6]) * line[1], perr[6] * line[1]])
        result.add_row([labels[2], 3*popt[5], popt[6], popt[7], 3*perr[5], perr[6], perr[7], (1 + popt[6]) * line[2], perr[6] * line[2]])

        width= popt[7]

    except:
        result.add_row([labels[0], np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan])
        result.add_row([labels[1], np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan])
        result.add_row([labels[2], np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan])


    plt.figure(figsize=(5*np.sqrt(2), 5))
    ax = plt.subplot(111)

    ax.plot(DATA['WAVE_REST'][mask_region], DATA['FLAMBDA_EXTCORR'][mask_region])
    ax.plot(DATA['WAVE_REST'][mask_region], DATA['FLAMBDA_EXTCORR'][mask_region]*0.1, color='tab:green')

    try:
        ax.plot(DATA['WAVE_REST'][mask_region], fitfunc(DATA['WAVE_REST'][mask_region], *popt))
    except:
        pass

    labels = ['[SII]-6716', '[SII]-6732']
    line = [lines[x] for x in labels]

    line_1 = min(line)
    line_2 = max(line)
    buffer_cont = 25
    buffer_lines = 7

    mask_region = (DATA['WAVE_REST'] > line_1 - buffer_cont) & (DATA['WAVE_REST'] < line_2 + buffer_cont)

    mask_line_1 = np.abs(DATA['WAVE_REST'] - line_1 ) < buffer_lines
    mask_line_2 = np.abs(DATA['WAVE_REST'] - line_2 ) < buffer_lines
    mask_lines  = np.logical_or(mask_line_1, mask_line_2)

    mask_cont   = np.logical_xor(mask_region, mask_lines)

    try:
        def fitfunc (x, A, B, C, D, A0, A1, REDSHIFT):

            W0 = width

            return 0*A * x**3 + B * x**2 + C * x + D + gauss_model(x, A0, (1 + REDSHIFT) * line[0], W0) +\
                                                       gauss_model(x, A1, (1 + REDSHIFT) * line[1], W0)

        pinit        = [0, 0, 0, 0,
                        np.nanmax(DATA['FLAMBDA_EXTCORR'][mask_region]),
                        np.nanmax(DATA['FLAMBDA_EXTCORR'][mask_region]),
                        0]

        from scipy.optimize import curve_fit
        popt, pcov     = curve_fit(fitfunc,
                                DATA['WAVE_REST'][mask_region],
                                DATA['FLAMBDA_EXTCORR'][mask_region],
                                sigma  = DATA['FLAMBDA_EXTCORR'][mask_region]*0.1,
                                p0     = pinit,
                                bounds = ([-np.inf, -np.inf, -np.inf, -np.inf, 0, 0, -0.1],
                                          [ np.inf,  np.inf,  np.inf,  np.inf, np.inf, np.inf, 0.1]))

        perr = np.sqrt(np.diag(pcov))

        result.add_row([labels[0], popt[4], popt[6], width, perr[4], perr[6], 0, (1 + popt[6]) * line[0], perr[6] * line[0]])
        result.add_row([labels[1], popt[5], popt[6], width, perr[5], perr[6], 0, (1 + popt[6]) * line[0], perr[6] * line[1]])

    except:
        result.add_row([labels[0], np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan])
        result.add_row([labels[1], np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan])


    plt.figure(figsize=(5*np.sqrt(2), 5))
    ax = plt.subplot(111)

    ax.plot(DATA['WAVE_REST'][mask_region], DATA['FLAMBDA_EXTCORR'][mask_region])
    ax.plot(DATA['WAVE_REST'][mask_region], DATA['FLAMBDA_EXTCORR'][mask_region]*0.1, color='tab:green')

    try:
        ax.plot(DATA['WAVE_REST'][mask_region], fitfunc(DATA['WAVE_REST'][mask_region], *popt))
    except:
        pass

    return result

In [ ]:
from astropy.io import ascii
Data = ascii.read(paths.MUSE / 'spectrum.txt')
print(Data)
result = measure_eml(Data, 0.0028)

In [ ]:
flux_oiii = [result['FLUX'][result['LINE'] == '[OIII]-5008'][0], result['FLUX_ERR'][result['LINE'] == '[OIII]-5008'][0]]
flux_hb   = [result['FLUX'][result['LINE'] == 'H-b'][0],         result['FLUX_ERR'][result['LINE'] == 'H-b'][0]]
flux_ha   = [result['FLUX'][result['LINE'] == 'H-a'][0],         result['FLUX_ERR'][result['LINE'] == 'H-a'][0]]
flux_nii  = [result['FLUX'][result['LINE'] == '[NII]-6583'][0],  result['FLUX_ERR'][result['LINE'] == '[NII]-6583'][0]]

_O3N2     = (flux_oiii[0] / flux_hb[0]) / (flux_nii[0] / flux_ha[0])
_O3N2_err = _O3N2* np.sqrt((flux_oiii[1]/flux_oiii[0])**2 + (flux_hb[1]/flux_hb[0])**2 + (flux_ha[1]/flux_ha[0])**2 + (flux_nii[1]/flux_nii[0])**2)

print('O3N2 = {val:.2f} +- {err:.2f}'.format(val=_O3N2, err=_O3N2_err))

O3N2 = lambda x: np.log10(_O3N2) - (0.281 - 4.765 * x - 2.268 * x**2)
value     = fsolve(O3N2, -0.1, maxfev=10000, xtol=1e-14)

O3N2 = lambda x: np.log10(_O3N2 + _O3N2_err) - (0.281 - 4.765 * x - 2.268 * x**2)
value_inf     = fsolve(O3N2, -0.1, maxfev=10000, xtol=1e-14)

O3N2 = lambda x: np.log10(_O3N2 - _O3N2_err) - (0.281 - 4.765 * x - 2.268 * x**2)
value_sup     = fsolve(O3N2, -0.1, maxfev=10000, xtol=1e-14)

sn2023zaw_03n2 = [10**value[0], 10**(value[0]) -10**value_inf[0], 10**value_sup[0] - 10**value[0]]

'Metallicity (unit: Z_sol) = {val:.2f} +{errp:.2f} - {errm:.2f}'.format(val=10**value[0], errp=10**(value[0]) -10**value_inf[0], errm=10**value_sup[0] - 10**value[0])

In [ ]:
flux_nii  = [result['FLUX'][result['LINE'] == '[NII]-6583'][0],  result['FLUX_ERR'][result['LINE'] == '[NII]-6583'][0]]
flux_sii = [result['FLUX'][result['LINE'] == '[SII]-6732'][0],  result['FLUX_ERR'][result['LINE'] == '[SII]-6732'][0]]
flux_sii_2 = [result['FLUX'][result['LINE'] == '[SII]-6716'][0],  result['FLUX_ERR'][result['LINE'] == '[SII]-6716'][0]]
flux_sii += flux_sii_2

y = np.log10(flux_nii[0]/flux_sii[0])+0.264*np.log10(flux_nii[0]/flux_ha[0])

print(10**(8.77+y-12))

In [ ]:
ratio_hghb            = 0.468
ratio_hahb            = 2.863

kha                    = 3.39747648#3.35477
khb                    = 4.57228234#4.62649
khg                    = 5.11955825#5.15326
khd                    = 5.37287672#5.42266

value_balmer          = eml_tools.balmer_decrement(result['FLUX'][result['LINE'] == 'H-a'][0],  result['FLUX'][result['LINE'] == 'H-b'][0], ratio_hahb, -2.5/(khb - kha))
value_balmer_errm     = eml_tools.balmer_decrement(result['FLUX'][result['LINE'] == 'H-a'][0] + result['FLUX_ERR'][result['LINE'] == 'H-a'][0], result['FLUX'][result['LINE'] == 'H-b'][0] - result['FLUX_ERR'][result['LINE'] == 'H-b'][0], ratio_hahb, -2.5/(khb - kha))
value_balmer_errp     = eml_tools.balmer_decrement(result['FLUX'][result['LINE'] == 'H-a'][0] - result['FLUX_ERR'][result['LINE'] == 'H-a'][0], result['FLUX'][result['LINE'] == 'H-b'][0] + result['FLUX_ERR'][result['LINE'] == 'H-b'][0], ratio_hahb, -2.5/(khb - kha))
value_balmer_err      = 2.5 / (khb-kha) / np.log(10) * np.sqrt((result['FLUX_ERR'][result['LINE'] == 'H-a'][0] / result['FLUX'][result['LINE'] == 'H-a'][0])**2 + (result['FLUX_ERR'][result['LINE'] == 'H-b'][0] / result['FLUX'][result['LINE'] == 'H-b'][0])**2)

#print(value_balmer, value_balmer-value_balmer_errm, value_balmer_errp-value_balmer, value_balmer_err)

# value_balmer = balmer_decrement_full(OBJECT, FILENAME, NITER=1E7, SIMPLE=True)

msg1 = 'E(B-V)_gas = {val:.02f} +- {err:.02f}'.format(val=value_balmer, err=value_balmer_err)
msg2 = 'E(B-V)_star = {val:.02f} +- {err:.02f}'.format(val=0.44*value_balmer, err=0.44*value_balmer_err)

msg3 = 'A_V,gas = {val:.02f} +- {err:.02f}'.format(val=4.05*value_balmer, err=4.05*value_balmer_err)
msg4 = 'A_V,star = {val:.02f} +- {err:.02f}'.format(val=4.05*0.44*value_balmer, err=4.05*0.44*value_balmer_err)

print(msg1), print(msg3)
print(msg2), print(msg4);

In [ ]:
# Uses Eq. 2 in Kennicutt (1998) and multiplies it by a factor of 0.63 to convert the Salpeter to a Chabrier IMF.
# Fudge factor from Madau & Dickinson (https://arxiv.org/abs/1403.0007)

flux_ha   = [result['FLUX'][result['LINE'] == 'H-a'][0], result['FLUX_ERR'][result['LINE'] == 'H-a'][0]]

NITER     = 10000

flux      = eml_tools.resample(flux_ha[0], flux_ha[1], flux_ha[1], NITER=NITER)
ebv       = eml_tools.resample(value_balmer, value_balmer_err, value_balmer_err, NITER=NITER) # Uses EBV_gas

# Luminosity distance and uncertaity taken from Paper
#luminosity_distance = np.random.normal(43.9, 3.1, NITER)*u.Mpc.to(u.cm)
luminosity_distance = 9*u.Mpc.to(u.cm)

sfr_uncorr          = 4 * np.pi * luminosity_distance**2 * flux * 7.9E-42 * 0.63
sfr_corr            = sfr_uncorr * eml_tools.ebvCor('ha', ebv=ebv, redlaw='cal', rv=4.05)

percentile_1        = np.percentile(sfr_uncorr, q=(50-68/2, 50, 50+68/2))
percentile_2        = np.percentile(sfr_corr,   q=(50-68/2, 50, 50+68/2))

msg1 = 'SFR (uncorrected, M_odot/yr) = {val:.03e} +{errp:.03e} -{errm:.03e}'.format(val=percentile_1[1], errp=percentile_1[2]-percentile_1[1], errm=percentile_1[1]-percentile_1[0])
msg2 = 'SFR (corrected, M_odot/yr) = {val:.03e} +{errp:.03e} -{errm:.03e}'.format(val=percentile_2[1], errp=percentile_2[2]-percentile_2[1], errm=percentile_2[1]-percentile_2[0])

print(msg1)
print(msg2)